# Phase 3: Re-run Fine-tuning (Class-Weighted Loss)

This notebook re-runs **only fine-tuning** (Stage 1 + Stage 2) for CAE, SimCLR, and MoCo in a fresh Colab session.
It does **not** re-run SSL pretraining — the pretrained encoder weights from the March 29 runs are copied from Drive.

## How checkpoint handoff works

Each fine-tuning script expects the **pretrained encoder weights** at a specific local path under `./output/`:

| Model | Script looks for | Copied from Drive |
|-------|-----------------|-------------------|
| CAE | `./output/models/exp_*/weights/VAE.weights.h5` | `runs/cae_20260329_114825/models/` |
| SimCLR | `./output/simclr/encoder_weights.h5` | `runs/simclr_20260329_122931/simclr/encoder_weights.weights.h5` |
| MoCo | `./output/models/moco/encoder_q_epoch*.weights.h5` | `runs/moco_20260329_024342/output/models/moco/` |

`./output/` is symlinked to a **new Phase 3 run directory** on Drive for each model.
Fine-tuning saves Stage 1 and Stage 2 checkpoints there automatically.

**Phase 3 change:** All three fine-tuning scripts now pass `class_weight` (capped at 3×) to `model.fit()`,
combined with the existing focal loss, to improve G5/G3 minority class recall.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')



import os

from datetime import datetime

from pathlib import Path



# ── Repo ─────────────────────────────────────────────────────────────────────

REPO_URL    = 'https://github.com/satvikkaul/SSL_Prostate_Cancer_Grading.git'

BRANCH      = 'abdur/phase3-colab-ready'

PROJECT_DIR = '/content/SSL_Prostate_Cancer_Grading'



# ── Drive paths ───────────────────────────────────────────────────────────────

DRIVE_ROOT  = '/content/drive/MyDrive/Prostate_SSL'

DATASET_ZIP = f'{DRIVE_ROOT}/dataset.zip'

RUNS_DIR    = f'{DRIVE_ROOT}/runs'



# ── Pretrain source runs (March 29) — encoder weights live here ───────────────

# DO NOT change these unless you re-ran pretraining

CAE_PRETRAIN_RUN    = f'{RUNS_DIR}/cae_20260329_114825'

SIMCLR_PRETRAIN_RUN = f'{RUNS_DIR}/simclr_20260329_122931'

MOCO_PRETRAIN_RUN   = f'{RUNS_DIR}/moco_20260329_024342'



os.environ['MPLCONFIGDIR'] = '/tmp/mplconfig'



print('PROJECT_DIR         =', PROJECT_DIR)

print('BRANCH              =', BRANCH)

print('RUNS_DIR            =', RUNS_DIR)

print()

print('Pretrain source runs:')

for label, path in [('CAE', CAE_PRETRAIN_RUN), ('SimCLR', SIMCLR_PRETRAIN_RUN), ('MoCo', MOCO_PRETRAIN_RUN)]:

    exists = '✅' if Path(path).exists() else '❌ MISSING'

    print(f'  {label:8s} {exists}  {path}')

print()

print('All run folders in RUNS_DIR:')

!ls -1 "$RUNS_DIR" | grep -E "(baseline|cae|moco|simclr)_202"


In [ ]:
# Clone the repo (gets the Phase 3 class-weight changes) and install dependencies
%cd /content
!rm -rf "$PROJECT_DIR"
!git clone -b "$BRANCH" "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"
!git branch --show-current
!git log -1 --oneline
print()

!pip install -q openpyxl scikit-learn pandas matplotlib pillow opencv-python
!python --version
!nvidia-smi | grep -E "GPU|CUDA|Driver" | head -5

In [ ]:
# Extract dataset from Drive to local SSD (much faster than reading from Drive during training)
%cd "$PROJECT_DIR"
!test -f "$DATASET_ZIP" || (echo "❌ Missing dataset.zip at $DATASET_ZIP" && false)

!rm -rf ./dataset
!cp "$DATASET_ZIP" ./dataset.zip
!unzip -q ./dataset.zip
!rm -f ./dataset.zip

# Verify or generate the split CSVs
import os
required = ['./dataset/Train.csv', './dataset/Test.csv',
            './dataset/TrainSplit.csv', './dataset/Val.csv',
            './dataset/Pretrain_Manifest.csv']
missing = [p for p in required if not os.path.exists(p)]
if missing:
    print('Generating missing CSVs:', missing)
    !python data/setup.py
else:
    print('✅ All dataset CSVs present')

!echo "Dataset files:" && ls ./dataset/*.csv

In [ ]:
# Verify all pretrained encoder checkpoints exist on Drive before starting any fine-tuning.
# These are the SSL pretraining outputs from the March 29 runs.
# Fine-tuning scripts load THESE weights — not the Stage 1 classifier checkpoints.
import glob
from pathlib import Path

checks = {
    # CAE: finetune_cae.py globs ./output/models/exp_*/weights/VAE.weights.h5
    'CAE encoder weights':
        sorted(glob.glob(f'{CAE_PRETRAIN_RUN}/models/exp_*/weights/VAE.weights.h5')),

    # SimCLR: finetune_simclr.py loads ./output/simclr/encoder_weights.h5
    # (Keras 3.x save_weights writes as .weights.h5 even when given .h5 path)
    'SimCLR encoder weights':
        sorted(glob.glob(f'{SIMCLR_PRETRAIN_RUN}/simclr/encoder_weights*.h5')),

    # MoCo: finetune_moco.py globs ./output/models/moco/encoder_q_epoch*.weights.h5
    'MoCo encoder weights':
        sorted(glob.glob(f'{MOCO_PRETRAIN_RUN}/output/models/moco/encoder_q_epoch*.weights.h5')),
}

# Also check existing Stage 1 + Stage 2 classifier checkpoints (Phase 2 reference)
stage_checks = {
    'CAE Stage 1 classifier':  f'{CAE_PRETRAIN_RUN}/best_model_stage1.keras',
    'CAE Stage 2 classifier':  f'{CAE_PRETRAIN_RUN}/best_model_fine_tuned.keras',
    'SimCLR Stage 1 classifier': f'{SIMCLR_PRETRAIN_RUN}/simclr/best_simclr_classifier.keras',
    'SimCLR Stage 2 classifier': f'{SIMCLR_PRETRAIN_RUN}/simclr/best_simclr_fine_tuned.keras',
    'MoCo Stage 2 classifier':   f'{MOCO_PRETRAIN_RUN}/output/moco/best_moco_fine_tuned.keras',
}

print("=" * 70)
print("PRETRAINED ENCODER WEIGHTS  (needed to run fine-tuning)")
print("=" * 70)
all_ok = True
for label, paths in checks.items():
    if paths:
        for p in paths:
            size_mb = Path(p).stat().st_size / 1e6
            print(f"  ✅ {label}")
            print(f"     {p}  ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ {label}  NOT FOUND — cannot run fine-tuning!")
        all_ok = False

print()
print("=" * 70)
print("EXISTING PHASE 2 CLASSIFIER CHECKPOINTS  (reference, not used here)")
print("=" * 70)
for label, path in stage_checks.items():
    exists = Path(path).exists()
    mark = '✅' if exists else '⚠️  missing'
    print(f"  {mark}  {label}")

if all_ok:
    print("\n✅ All encoder weights found — ready to run Phase 3 fine-tuning")
else:
    print("\n❌ Some encoder weights are missing. Check that RUNS_DIR is correct.")

In [ ]:
# Inspect how each fine-tuning script locates its encoder weights.
# This cell shows the exact lines in each script that control the checkpoint path.
%cd "$PROJECT_DIR"
print("=" * 70)
print("CAE — finetune_cae.py: how it finds encoder weights")
print("=" * 70)
!grep -n "WEIGHTS_PATH\|weights auto\|auto-detect\|exp_" training/cae/finetune_cae.py | head -20

print()
print("=" * 70)
print("SimCLR — finetune_simclr.py: how it finds encoder weights")
print("=" * 70)
!grep -n "ENCODER_WEIGHTS\|load_weights\|encoder_weights" training/simclr/finetune_simclr.py | head -15

print()
print("=" * 70)
print("MoCo — finetune_moco.py: how it finds encoder weights")
print("=" * 70)
!grep -n "DEFAULT_CHECKPOINT_GLOB\|checkpoint\|encoder_q" training/moco/finetune_moco.py | head -15

print()
print("=" * 70)
print("Phase 3 class-weight changes: confirming they're in the repo")
print("=" * 70)
!grep -n "class_weight\|capped at" training/cae/finetune_cae.py | head -10
!grep -n "class_weight\|capped at" training/simclr/finetune_simclr.py | head -10
!grep -n "class_weight\|capped at" training/moco/finetune_moco.py | head -10

## CAE Fine-tuning (Phase 3)

**What happens:**
1. A new `cae_phase3_TIMESTAMP` run directory is created on Drive
2. `./output` is symlinked to it
3. CAE autoencoder weights are copied from `cae_20260329_114825/models/` into `./output/models/` — this is what `--weights auto` detects
4. `finetune_cae.py` runs Stage 1 (frozen encoder, 50 epochs) then Stage 2 (end-to-end, 30 epochs)
5. Both stages use focal loss **+ class weights capped at 3×** (Phase 3 change)
6. Checkpoints saved: `./output/best_model_stage1.keras`, `./output/best_model_fine_tuned.keras`

In [ ]:
import shutil, os, glob
from datetime import datetime
from pathlib import Path

# ── 1. Create new Phase 3 run dir on Drive ────────────────────────────────────
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
CAE_P3_RUN = f'{RUNS_DIR}/cae_phase3_{ts}'
os.makedirs(CAE_P3_RUN, exist_ok=True)
print(f'CAE Phase 3 run dir: {CAE_P3_RUN}')

# ── 2. Symlink ./output → Drive run dir ──────────────────────────────────────
%cd "$PROJECT_DIR"
if os.path.islink('./output') or os.path.exists('./output'):
    if os.path.islink('./output'):
        os.unlink('./output')
    else:
        shutil.rmtree('./output')
os.symlink(CAE_P3_RUN, './output')
print(f'Linked ./output → {CAE_P3_RUN}')

# ── 3. Copy CAE encoder weights → ./output/models/  ──────────────────────────
# finetune_cae.py --weights auto globs: ./output/models/exp_*/weights/VAE.weights.h5
src_models = Path(CAE_PRETRAIN_RUN) / 'models'
dst_models = Path('./output/models')

if src_models.exists():
    shutil.copytree(str(src_models), str(dst_models))
    found = sorted(glob.glob('./output/models/exp_*/weights/VAE.weights.h5'))
    print(f'Copied encoder weights: {found}')
    if not found:
        raise RuntimeError('No VAE.weights.h5 found — check CAE_PRETRAIN_RUN path')
else:
    raise RuntimeError(f'CAE models dir not found: {src_models}')

print('\n✅ Setup complete — starting CAE fine-tuning')
print('   Stage 1: 50 epochs (frozen encoder)')
print('   Stage 2: 30 epochs (end-to-end)')
print()

# ── 4. Run fine-tuning ────────────────────────────────────────────────────────
CAE_BATCH    = 32
CAE_S1_EPOCHS = 50
CAE_S2_EPOCHS = 30

cmd = (f'python training/cae/finetune_cae.py'
       f' --epochs_stage1 {CAE_S1_EPOCHS}'
       f' --epochs_stage2 {CAE_S2_EPOCHS}'
       f' --batch_size {CAE_BATCH}'
       f' --weights auto')
print(cmd)
!$cmd

# ── 5. Confirm outputs were saved ────────────────────────────────────────────
print('\nCAE Phase 3 outputs:')
for pattern in ['best_model_stage1.keras', 'best_model_fine_tuned.keras',
                'cae/best_cae_classifier.keras', 'cae/best_cae_fine_tuned.keras']:
    p = Path('./output') / pattern
    if p.exists():
        mb = p.stat().st_size / 1e6
        print(f'  ✅ {pattern}  ({mb:.1f} MB)')
    else:
        print(f'  ❌ {pattern}  MISSING')

print(f'\nCAE_P3_RUN = "{CAE_P3_RUN}"')
print('(you will need this path to update the evaluation notebook)')

## SimCLR Fine-tuning (Phase 3)

**What happens:**
1. A new `simclr_phase3_TIMESTAMP` run directory is created on Drive
2. `./output` symlink is re-pointed to it
3. SimCLR encoder weights are copied from `simclr_20260329_122931/simclr/encoder_weights.weights.h5` → `./output/simclr/encoder_weights.h5`
   - Keras 3.x saves weights with `.weights.h5` suffix but `load_weights` accepts either form; both copies are placed just in case
4. `finetune_simclr.py` runs Stage 1 (50 epochs) then Stage 2 (30 epochs)
5. Checkpoints saved: `./output/simclr/best_simclr_classifier.keras`, `./output/simclr/best_simclr_fine_tuned.keras`

In [ ]:
# Optional: refresh the existing Colab clone to the latest commit on BRANCH
%cd "$PROJECT_DIR"

!git fetch origin "$BRANCH"

!git checkout "$BRANCH"

!git reset --hard "origin/$BRANCH"

!git log -1 --oneline

print("\nRepo refreshed. You can now run the SimCLR fine-tuning cell without repeating full setup.")
print("This refresh updates code only; it does not delete the Drive-backed ./output symlink or the extracted ./dataset folder.")

### Optional: Refresh Repo Before SimCLR



If you have already cloned the repo in this Colab runtime and pushed a fix afterward, run the next cell to update the current clone to the latest commit on `BRANCH`.



This refreshes code only and does not re-run the full setup flow.


In [ ]:
import shutil, os, glob
from datetime import datetime
from pathlib import Path

# ── 1. Create new Phase 3 run dir on Drive ────────────────────────────────────
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
SIMCLR_P3_RUN = f'{RUNS_DIR}/simclr_phase3_{ts}'
os.makedirs(SIMCLR_P3_RUN, exist_ok=True)
print(f'SimCLR Phase 3 run dir: {SIMCLR_P3_RUN}')

# ── 2. Re-symlink ./output ────────────────────────────────────────────────────
%cd "$PROJECT_DIR"
if os.path.islink('./output') or os.path.exists('./output'):
    if os.path.islink('./output'):
        os.unlink('./output')
    else:
        shutil.rmtree('./output')
os.symlink(SIMCLR_P3_RUN, './output')
print(f'Linked ./output → {SIMCLR_P3_RUN}')

# ── 3. Copy SimCLR encoder weights → ./output/simclr/ ───────────────────────
# finetune_simclr.py should load the canonical .weights.h5 file on Keras 3.x
# We also copy the legacy .h5 name for compatibility, but pass the canonical path explicitly

src_simclr = Path(SIMCLR_PRETRAIN_RUN) / 'simclr'
dst_simclr = Path('./output/simclr')
dst_simclr.mkdir(parents=True, exist_ok=True)

src_weights = list(src_simclr.glob('encoder_weights.weights.h5'))
if not src_weights:
    src_weights = list(src_simclr.glob('encoder_weights*.h5'))
if not src_weights:
    raise RuntimeError(f'SimCLR encoder weights not found in {src_simclr}')

src_w = sorted(src_weights)[0]
print(f'Source encoder weights: {src_w}  ({src_w.stat().st_size/1e6:.1f} MB)')

canonical_weights = dst_simclr / 'encoder_weights.weights.h5'
legacy_weights = dst_simclr / 'encoder_weights.h5'
shutil.copy2(src_w, canonical_weights)
shutil.copy2(src_w, legacy_weights)
print(f'Copied to: {canonical_weights}')
print(f'Copied compatibility alias: {legacy_weights}')

print('\n✅ Setup complete — starting SimCLR fine-tuning')
print()

# ── 4. Run fine-tuning ────────────────────────────────────────────────────────
SIMCLR_BATCH = 32
SIMCLR_S1_EPOCHS = 50
SIMCLR_S2_EPOCHS = 30

cmd = (f'python training/simclr/finetune_simclr.py'
       f' --epochs_stage1 {SIMCLR_S1_EPOCHS}'
       f' --epochs_stage2 {SIMCLR_S2_EPOCHS}'
       f' --batch_size {SIMCLR_BATCH}'
       f' --encoder_weights {canonical_weights}')
print(cmd)
!$cmd

# ── 5. Confirm outputs ────────────────────────────────────────────────────────
print('\nSimCLR Phase 3 outputs:')
for fname in ['simclr/best_simclr_classifier.keras', 'simclr/best_simclr_fine_tuned.keras',
              'simclr/simclr_classifier_final.keras']:
    p = Path('./output') / fname
    if p.exists():
        mb = p.stat().st_size / 1e6
        print(f'  ✅ {fname}  ({mb:.1f} MB)')
    else:
        print(f'  ❌ {fname}  MISSING')

print(f'\nSIMCLR_P3_RUN = "{SIMCLR_P3_RUN}"')
print('(you will need this path to update the evaluation notebook)')

## MoCo Fine-tuning (Phase 3)

**What happens:**
1. A new `moco_phase3_TIMESTAMP` run directory is created on Drive
2. `./output` symlink is re-pointed to it
3. MoCo encoder weights are copied from `moco_20260329_024342/output/models/moco/encoder_q_epoch*.weights.h5` → `./output/models/moco/`
   - `finetune_moco.py` globs `./output/models/moco/encoder_q_epoch*.weights.h5` and picks the latest
4. `finetune_moco.py` runs Stage 1 (30 epochs) then Stage 2 (10 epochs)
5. Checkpoints saved: `./output/moco/best_moco_classifier.keras`, `./output/moco/best_moco_fine_tuned.keras`, `./output/moco/best_moco_overall.keras`

In [ ]:
import shutil, os, glob
from datetime import datetime
from pathlib import Path

# ── 1. Create new Phase 3 run dir on Drive ────────────────────────────────────
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
MOCO_P3_RUN = f'{RUNS_DIR}/moco_phase3_{ts}'
os.makedirs(MOCO_P3_RUN, exist_ok=True)
print(f'MoCo Phase 3 run dir: {MOCO_P3_RUN}')

# ── 2. Re-symlink ./output ────────────────────────────────────────────────────
%cd "$PROJECT_DIR"
if os.path.islink('./output') or os.path.exists('./output'):
    if os.path.islink('./output'):
        os.unlink('./output')
    else:
        shutil.rmtree('./output')
os.symlink(MOCO_P3_RUN, './output')
print(f'Linked ./output → {MOCO_P3_RUN}')

# ── 3. Copy MoCo encoder weights → ./output/models/moco/ ─────────────────────
# finetune_moco.py globs: ./output/models/moco/encoder_q_epoch*.weights.h5
# picks the latest by filename (epoch080 in our case)
src_moco = Path(MOCO_PRETRAIN_RUN) / 'output' / 'models' / 'moco'
dst_moco = Path('./output/models/moco')
dst_moco.mkdir(parents=True, exist_ok=True)

src_weights = sorted(src_moco.glob('encoder_q_epoch*.weights.h5'))
if not src_weights:
    raise RuntimeError(f'MoCo encoder weights not found in {src_moco}')

for w in src_weights:
    shutil.copy2(w, dst_moco / w.name)
    mb = w.stat().st_size / 1e6
    print(f'Copied: {w.name}  ({mb:.1f} MB)')

copied = sorted(glob.glob('./output/models/moco/encoder_q_epoch*.weights.h5'))
print(f'\nDetectable by script: {[Path(p).name for p in copied]}')

print('\n✅ Setup complete — starting MoCo fine-tuning')
print()

# ── 4. Run fine-tuning ────────────────────────────────────────────────────────
MOCO_BATCH    = 32
MOCO_S1_EPOCHS = 30
MOCO_S2_EPOCHS = 10

# Use the latest encoder checkpoint (epoch080)
latest_ckpt = copied[-1]
cmd = (f'python training/moco/finetune_moco.py'
       f' --checkpoint {latest_ckpt}'
       f' --epochs_stage1 {MOCO_S1_EPOCHS}'
       f' --epochs_stage2 {MOCO_S2_EPOCHS}'
       f' --batch_size {MOCO_BATCH}')
print(cmd)
!$cmd

# ── 5. Confirm outputs ────────────────────────────────────────────────────────
print('\nMoCo Phase 3 outputs:')
for fname in ['moco/best_moco_classifier.keras', 'moco/best_moco_fine_tuned.keras',
              'moco/best_moco_overall.keras', 'moco/moco_classifier_final.keras']:
    p = Path('./output') / fname
    if p.exists():
        mb = p.stat().st_size / 1e6
        print(f'  ✅ {fname}  ({mb:.1f} MB)')
    else:
        print(f'  ❌ {fname}  MISSING')

print(f'\nMOCO_P3_RUN = "{MOCO_P3_RUN}"')

In [ ]:
# Verify all Phase 3 output checkpoints are on Drive and print their file sizes/timestamps
from pathlib import Path
from datetime import datetime

print("=" * 70)
print("PHASE 3 OUTPUT CHECKPOINT SUMMARY")
print("=" * 70)

phase3_outputs = {
    'CAE Stage 1':   (CAE_P3_RUN,    'best_model_stage1.keras'),
    'CAE Stage 2':   (CAE_P3_RUN,    'best_model_fine_tuned.keras'),
    'SimCLR Stage 1':(SIMCLR_P3_RUN, 'simclr/best_simclr_classifier.keras'),
    'SimCLR Stage 2':(SIMCLR_P3_RUN, 'simclr/best_simclr_fine_tuned.keras'),
    'MoCo Stage 2':  (MOCO_P3_RUN,   'moco/best_moco_fine_tuned.keras'),
}

all_saved = True
for label, (run_dir, rel_path) in phase3_outputs.items():
    p = Path(run_dir) / rel_path
    if p.exists():
        stat = p.stat()
        mb = stat.st_size / 1e6
        mtime = datetime.fromtimestamp(stat.st_mtime).strftime('%H:%M:%S')
        print(f'  ✅ {label:<18s}  {mb:6.1f} MB   saved at {mtime}')
        print(f'     {p}')
    else:
        print(f'  ❌ {label:<18s}  MISSING — {p}')
        all_saved = False

print()
if all_saved:
    print("✅ All Phase 3 checkpoints saved to Drive")
else:
    print("⚠️  Some checkpoints are missing — check the fine-tuning cell output above")

In [ ]:
# Print the exact Phase 3 evaluation command so this notebook is self-contained

from pathlib import Path



print("=" * 70)

print("PHASE 3 EVALUATION COMMAND")

print("=" * 70)



cae_run_name = Path(CAE_P3_RUN).name

simclr_run_name = Path(SIMCLR_P3_RUN).name

moco_run_name = Path(MOCO_P3_RUN).name



phase3_eval_cmd = (

    "python phase3_comparison/evaluate_phase3_comparison.py "

    f"--runs-dir {RUNS_DIR} "

    "--dataset-root ./dataset "

    f"--cae-run {cae_run_name} "

    f"--simclr-run {simclr_run_name} "

    f"--moco-run {moco_run_name}"

)



print(phase3_eval_cmd)

print()

print("Outputs will be saved to:")

print(f"  {RUNS_DIR}/phase3_comparison/stage_comparison_table.csv")

print(f"  {RUNS_DIR}/phase3_comparison/stage_comparison_detailed_results.json")

print()

print("Phase 2 outputs remain untouched in:")

print(f"  {RUNS_DIR}/phase2_comparison/")


## Evaluate Phase 3 Results



This notebook now evaluates the new Phase 3 fine-tuned checkpoints directly, without needing the older Phase 2 evaluation notebook.



- Uses `phase3_comparison/evaluate_phase3_comparison.py`

- Preserves existing `phase2_comparison/` outputs

- Writes new outputs to `RUNS_DIR/phase3_comparison/`


In [ ]:
# Run Phase 3 evaluation and save outputs to a separate phase3_comparison folder

from pathlib import Path



cae_run_name = Path(CAE_P3_RUN).name

simclr_run_name = Path(SIMCLR_P3_RUN).name

moco_run_name = Path(MOCO_P3_RUN).name



cmd = (

    "python phase3_comparison/evaluate_phase3_comparison.py "

    f"--runs-dir {RUNS_DIR} "

    "--dataset-root ./dataset "

    f"--cae-run {cae_run_name} "

    f"--simclr-run {simclr_run_name} "

    f"--moco-run {moco_run_name}"

)



print(cmd)

!$cmd



print("\nSaved files:")

!ls -lh "$RUNS_DIR/phase3_comparison"
